In [1]:
install.packages("table1")

Installing package into ‘/home/jupyter/.R/library’
(as ‘lib’ is unspecified)



In [2]:
library(bigrquery)
library(knitr)
library(tidyverse)
library(ggplot2)
library("table1")
library("IRdisplay")
bq_auth() 

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.2     ✔ readr     2.1.4
✔ forcats   1.0.0     ✔ stringr   1.5.0
✔ ggplot2   3.4.4     ✔ tibble    3.2.1
✔ lubridate 1.9.2     ✔ tidyr     1.3.0
✔ purrr     1.0.1     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors

Attaching package: ‘table1’


The following objects are masked from ‘package:base’:

    units, units<-




In [3]:
data <-bq_dataset_query(query="
SELECT *

 FROM `yhcr-prd-phm-bia-core.CB_1935_AK.src_bmbc_EYFSP_PSEG07IMD`"
                       ,x="yhcr-prd-phm-bia-core.CB_1935_AK")

EYFSPPSEG07 <- bq_table_download(data) 

In [4]:
data1 <-bq_dataset_query(query="
select round(avg(count1),5)

from(

SELECT person_id,count(person_id) count1

 FROM `yhcr-prd-phm-bia-core.CB_1935_AK.src_bmbc_EYFSP_PSEG07`

 group by person_id)base"
                       ,x="yhcr-prd-phm-bia-core.CB_1935_AK")

EYFSPPSEG07AVG <- bq_table_download(data1) 

In [5]:
EYFSPPSEG07$Gender[(EYFSPPSEG07$Gender) == "F"] <- "Female"
EYFSPPSEG07$Gender[(EYFSPPSEG07$Gender) == "M"] <- "Male"
EYFSPPSEG07$Gender[(EYFSPPSEG07$Gender) == "U"] <- "Unknown"

In [6]:
display_jupyter <- function(x) {
  css <- system.file("table1_defaults_1.0/table1_defaults.css", package="table1")
  css <- paste(readLines(css), collapse="\n")
  x <- htmltools::tagList(htmltools::tags$style(css), htmltools::tags$div(class="Rtable1", x))
  IRdisplay::display_html(as.character(x))
}

In [7]:
label(EYFSPPSEG07$AcademicYear)      <- "Academic Year"
label(EYFSPPSEG07$IMD)      <- "Average IMD"

label(EYFSPPSEG07$FSMEligible)      <- "FSM Eligiblity"
label(EYFSPPSEG07$ethnic_group)      <- "Ethnic Group"
label(EYFSPPSEG07$Gender)      <- "Gender"

In [8]:
 x<-table1(~ EYFSPPSEG07$AcademicYear + EYFSPPSEG07$FSMEligible +EYFSPPSEG07$Gender+ EYFSPPSEG07$ethnic_group+ EYFSPPSEG07$IMD|EYFSPPSEG07$score,data=EYFSPPSEG07
          ,justify=c("left", "left", rep("center", 5)),format_number = TRUE
          ,caption = "Table presenting PSEG07 - Managing feelings and behavior results from the Early years foundation stage profile (EYFSP)  "
    
         )

In [11]:
display_jupyter(x)

,1 - Emerging(N=13164),2 - Expected level(N=52087),3 - Exceeded(N=10324),A - Exemption not assessed (N=180),No Data(N=23),Overall(N=75778)
Academic Year,,,,,,
2012/2013,2603 (19.8%),7554 (14.5%),1146 (11.1%),28 (15.6%),0 (0%),11331 (15.0%)
2013/2014,2406 (18.3%),7523 (14.4%),1413 (13.7%),18 (10.0%),0 (0%),11360 (15.0%)
2014/2015,1883 (14.3%),7733 (14.8%),1494 (14.5%),31 (17.2%),0 (0%),11141 (14.7%)
2015/2016,1591 (12.1%),7607 (14.6%),1708 (16.5%),0 (0%),23 (100%),10929 (14.4%)
2016/2017,1644 (12.5%),7465 (14.3%),1625 (15.7%),19 (10.6%),0 (0%),10753 (14.2%)
2017/2018,1598 (12.1%),7280 (14.0%),1436 (13.9%),52 (28.9%),0 (0%),10366 (13.7%)
2018/2019,1439 (10.9%),6925 (13.3%),1502 (14.5%),32 (17.8%),0 (0%),9898 (13.1%)
FSM Eligiblity,,,,,,
Yes,3552 (27.0%),9861 (18.9%),1148 (11.1%),36 (20.0%),6 (26.1%),14603 (19.3%)


In [9]:
OverallTrend1 <- EYFSPPSEG07 %>% select(1,5,7)

PersonalSocial1 <- OverallTrend1 %>%
  group_by(AcademicYear,score) 

PS1<- PersonalSocial1 %>% count(AcademicYear,score)


PS1<- group_by(PS1, AcademicYear) %>% mutate(percent = (n/sum(n))*100)


#PS1<-unique(filter(PS1,result == ))

#PS1<- group_by(PS1, AcademicYear) %>% mutate(percent = (n/sum(n))*100)

PS1<-unique(filter(PS1,score == "2 - Expected level" |score == "3 - Exceeded" ))
PS1 <- PS1 %>% select(1,4)

PS1<- group_by(PS1, AcademicYear) %>% mutate(PctPerYear =(sum(percent)))
PS1 <- PS1 %>% select(1,3)

PS1 <- distinct(PS1)


OVERALL<-ggplot(PS1 , aes(x = AcademicYear, y = PctPerYear,group = 1)) +
#scale_y_continuous( limits=c(0,1))+
  geom_line() +
  geom_point() +
   ggtitle(expression(atop("COS-EY 1: 1.3 Emotional & social development",atop("PSEG07 - Managing feelings and behavior results from the Early years foundation stage profile (EYFSP)"), atop(italic("percentage of people achieveing atleast 'Expected' grade"), "")))) +


labs(
       x = "Academic Year",
       y = "Percentage of students per year") +
scale_y_continuous(labels = function(x) paste0(x, "%"))+
               theme(panel.border = element_blank(),# panel.grid.major = element_blank(),
#panel.grid.minor = element_blank(), 
      axis.line = element_line(colour = "black"),axis.text.x = element_text(angle = 90, vjust = 0.5, hjust=1))+
geom_text(aes(label = round(PctPerYear,0)), hjust = 0.5,  vjust = -1) +
                   #+
  theme_minimal()

In [10]:
OverallTrend1 <- EYFSPPSEG07 %>% select(1,2,5,7)

PersonalSocial1 <- OverallTrend1 %>%
  group_by(AcademicYear,score,ethnic_group) 

PS1<- PersonalSocial1 %>% count(AcademicYear,score,ethnic_group)


PS1<- group_by(PS1, AcademicYear,ethnic_group) %>% mutate(percent = (n/sum(n))*100)


#PS1<-unique(filter(PS1,result == ))

#PS1<- group_by(PS1, AcademicYear,ethnic_group) %>% mutate(percent = (n/sum(n))*100)

PS1<-unique(filter(PS1,score == "2 - Expected level" |score == "3 - Exceeded" ))
PS1 <- PS1 %>% select(1,3,5)

PS1<- group_by(PS1, AcademicYear,ethnic_group) %>% mutate(PctPerYear =round(sum(percent)),2)
PS1 <- PS1 %>% select(1,2,4)

PS1 <- distinct(PS1)

ethnicity <- ggplot(PS1 , aes(x = AcademicYear, y = PctPerYear,group = ethnic_group,color = ethnic_group)) +
#scale_y_continuous( limits=c(0,1))+
  geom_line() +
  geom_point() +
   ggtitle(expression(atop("COS-EY 1: 1.3 Emotional & social development", atop(italic("percentage of people achieveing atleast 'Expected' grade"),
                                                                                  atop(italic("By ethnic group")))))) +
  labs(
       x = "Academic Year",
       y = "Percentage of students per year") +
scale_y_continuous(labels = function(x) paste0(x, "%"))+
               theme(panel.border = element_blank(),# panel.grid.major = element_blank(),
#panel.grid.minor = element_blank(), 
      axis.line = element_line(colour = "black"),axis.text.x = element_text(angle = 45, vjust = 1, hjust=1))+
geom_text(aes(label = round(PctPerYear,0)), hjust = 0.75,  vjust = -2) +
                   theme_minimal()+
  theme(axis.text.x = element_text(angle = 90, vjust = 0.5, hjust = 1))

In [11]:
OverallTrend1 <- EYFSPPSEG07 %>% select(1,4,5,7)

PersonalSocial1 <- OverallTrend1 %>%
  group_by(AcademicYear,score,Gender) 

PS1<- PersonalSocial1 %>% count(AcademicYear,score,Gender)


PS1<- group_by(PS1, AcademicYear,Gender) %>% mutate(percent = (n/sum(n))*100)
#PS1<-unique(filter(PS1,result == ))

#PS1<- group_by(PS1, AcademicYear,ethnic_group) %>% mutate(percent = (n/sum(n))*100)

PS1<-unique(filter(PS1,score == "2 - Expected level" |score == "3 - Exceeded" ))
PS1 <- PS1 %>% select(1,3,5)

PS1<- group_by(PS1, AcademicYear,Gender) %>% mutate(PctPerYear =round(sum(percent)),2)
PS1 <- PS1 %>% select(1,2,4)

PS1 <- distinct(PS1)
PS1 <-PS1<-unique(filter(PS1,Gender == "Female" |Gender == "Male" ))

gender <- ggplot(PS1 , aes(x = AcademicYear, y = PctPerYear,group = Gender,color = Gender)) +
#scale_y_continuous( limits=c(0,1))+
  geom_line() +
  geom_point() +
   ggtitle(expression(atop("COS-EY 1: 1.3 Emotional & social development", atop(italic("percentage of people achieveing atleast 'Expected' grade"),
                                                                                  atop(italic("By Gender")))))) +
  labs(
       x = "Academic Year",
       y = "Percentage of students per year") +
scale_y_continuous(labels = function(x) paste0(x, "%"))+
               theme(panel.border = element_blank(),# panel.grid.major = element_blank(),
#panel.grid.minor = element_blank(), 
      axis.line = element_line(colour = "black"),axis.text.x = element_text(angle = 45, vjust = 1, hjust=1))+
geom_text(aes(label = round(PctPerYear,0)), hjust = 0.75,  vjust = -1) +
                   theme_minimal()+
  theme(axis.text.x = element_text(angle = 90, vjust = 0.5, hjust = 1))

In [12]:
OverallTrend1 <- EYFSPPSEG07 %>% select(1,10,5,7)

PersonalSocial1 <- OverallTrend1 %>%
  group_by(AcademicYear,score,IMD) 

PS1<- PersonalSocial1 %>% count(AcademicYear,score,IMD)


PS1<- group_by(PS1, AcademicYear,IMD) %>% mutate(percent = (n/sum(n))*100)
#PS1<-unique(filter(PS1,result == ))

#PS1<- group_by(PS1, AcademicYear,ethnic_group) %>% mutate(percent = (n/sum(n))*100)

PS1<-unique(filter(PS1,score == "2 - Expected level" |score == "3 - Exceeded" ))
PS1 <- PS1 %>% select(1,3,5)

PS1<- group_by(PS1, AcademicYear,IMD) %>% mutate(PctPerYear =round(sum(percent)),2)
PS1 <- PS1 %>% select(1,2,4)

PS1 <- distinct(PS1)

 IMD <- ggplot(PS1 , aes(x = AcademicYear, y = PctPerYear,group = IMD,color = IMD)) +
#scale_y_continuous( limits=c(0,1))+
  geom_line() +
  geom_point() +
   ggtitle(expression(atop("COS-EY 1: 1.3 Emotional & social development", atop(italic("percentage of people achieveing atleast 'Expected' grade"),
                                                                                  atop(italic("By IMD")))))) +
  labs(
       x = "Academic Year",
       y = "Percentage of students per year") +
scale_y_continuous(labels = function(x) paste0(x, "%"))+
               theme(panel.border = element_blank(),# panel.grid.major = element_blank(),
#panel.grid.minor = element_blank(), 
      axis.line = element_line(colour = "black"),axis.text.x = element_text(angle = 45, vjust = 1, hjust=1))+
geom_text(aes(label = round(PctPerYear,0)), hjust = 0.75,  vjust = -1) +
                   theme_minimal()+
  theme(axis.text.x = element_text(angle = 90, vjust = 0.5, hjust = 1))

In [13]:
library(grid)
#library(gridtext)
library(gridExtra)
plot <- list(OVERALL, gender, IMD, ethnicity)
pdf('1.3 Emotional & social development2.pdf',width=10, height=10)
  title <- "Plots for 1.3 Emotional & social development : PSEG07 - Managing feelings and behavior"
grid.text(title) 
plot
dev.off()
#


Attaching package: ‘gridExtra’


The following object is masked from ‘package:dplyr’:

    combine




[[1]]

[[2]]

[[3]]

[[4]]


png 
  2